In [ ]:
# One-cell external asset-forensics runner (all outputs remain private in Google Drive)
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, pathlib, json
from IPython.display import display, clear_output

REPO = "https://github.com/ZzwahwahzZ/Clash-of-Legends.git"
BRANCH = "tools/colab-drive-asset-transfer"
WORK = pathlib.Path("/content/clash-of-legends-forensics")
OUT_ROOT = pathlib.Path("/content/drive/MyDrive/ClashOfLegends_Forensics")

print("Installing independent open tools: 7-Zip, libarchive, binwalk, ffmpeg, Blender...")
subprocess.run(["bash","-lc","apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq p7zip-full libarchive-tools binwalk ffmpeg blender"],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","kaitaistruct","Pillow","texture2ddecoder","ipyfilechooser"],check=True)
import ipywidgets as widgets
from ipyfilechooser import FileChooser

if WORK.exists():
    subprocess.run(["git","-C",str(WORK),"fetch","origin",BRANCH],check=True)
    subprocess.run(["git","-C",str(WORK),"checkout",BRANCH],check=True)
    subprocess.run(["git","-C",str(WORK),"pull","--ff-only","origin",BRANCH],check=True)
else:
    subprocess.run(["git","clone","--depth","1","--branch",BRANCH,REPO,str(WORK)],check=True)



def run_pipeline(selected_source):
    if True:
        source = pathlib.Path(selected_source).expanduser().resolve()
        if not source.is_file() or source.suffix.lower() not in {'.zip','.7z','.rar','.tar','.tgz','.apk','.obb'}:
            raise FileNotFoundError(f'اختاري ملف أرشيف صالحًا: {source}')
        destination = OUT_ROOT / source.stem
        destination.mkdir(parents=True,exist_ok=True)
        scanner = WORK / "tools/asset_forensics/run_forensics.py"
        print("المصدر:", source, flush=True)
        print('بدء الفحص والاستخراج الآن...', flush=True)
        print("مكان النتائج الخاص:", destination)
        cmd = [sys.executable,str(scanner),str(source),"--out",str(destination)]
        result = subprocess.run(cmd,text=True,check=False)
        print('انتهى ماسح الأرشيفات؛ نتحقق من التقارير ونكمل المراحل الممكنة.', flush=True)
        print("\nExit code:",result.returncode)
        report = destination / "forensic-report.json"
        candidates = destination / "resource-candidates.csv"
        print("التقرير:",report)
        print("فهرس الموارد:",candidates)
        print("الملفات المستخرجة:",destination / "extracted")
        tpi_extractor = WORK / "tools/asset_forensics/extract_mobile_tpi.py"
        tpi_output = destination / "extracted-mobile-tpi"
        subprocess.run([sys.executable,str(tpi_extractor),str(destination/"extracted/root"),"--out",str(tpi_output)],check=True)
        tpi_report = json.loads((tpi_output/"tpi-extraction-report.json").read_text(encoding="utf-8"))
        print("\\nاستخراج حاويات TPI/TPD:",tpi_report["extracted_count"],"مورد /",tpi_report["tpi_count"],"حزمة /",len(tpi_report["errors"]),"خطأ")
        print("تقرير TPI:",tpi_output/"tpi-extraction-report.json")
        mjp_decoder = WORK / "tools/asset_forensics/decode_mobile_mjp.py"
        mjp_output = destination / "decoded-mobile-images"
        subprocess.run([sys.executable,str(mjp_decoder),str(destination/"extracted/root"),"--out",str(mjp_output)],check=False)
        if (mjp_output/"mjp-decode-report.json").exists():
            mjp_report = json.loads((mjp_output/"mjp-decode-report.json").read_text(encoding="utf-8"))
            print("\\nصور MJP:",mjp_report["decoded_count"],"تم فكها /",mjp_report["found"],"ملف /",len(mjp_report["errors"]),"أخطاء")
        mjp_tpi_output = destination / "decoded-mobile-tpi-images"
        subprocess.run([sys.executable,str(mjp_decoder),str(tpi_output),"--out",str(mjp_tpi_output)],check=False)
        if (mjp_tpi_output/"mjp-decode-report.json").exists():
            mjp_tpi_report = json.loads((mjp_tpi_output/"mjp-decode-report.json").read_text(encoding="utf-8"))
            print("صور MJP داخل TPI:",mjp_tpi_report["decoded_count"],"تم فكها /",mjp_tpi_report["found"],"ملف /",len(mjp_tpi_report["errors"]),"أخطاء")
        pvr_decoder = WORK / "tools/asset_forensics/decode_mobile_pvr.py"
        pvr_output = destination / "decoded-mobile-pvr"
        subprocess.run([sys.executable,str(pvr_decoder),str(tpi_output),"--out",str(pvr_output)],check=False)
        if (pvr_output/"pvr-decode-report.json").exists():
            pvr_report = json.loads((pvr_output/"pvr-decode-report.json").read_text(encoding="utf-8"))
            print("خامات PVR:",pvr_report["decoded_count"],"تم فكها /",pvr_report["found"],"ملف /",len(pvr_report["errors"]),"أخطاء")
            print("تقرير PVR:",pvr_output/"pvr-decode-report.json")
        processor = WORK / "tools/asset_forensics/process_standard_assets.py"
        converted = destination / "converted"
        subprocess.run([sys.executable,str(processor),str(destination/"extracted/root"),"--out",str(converted)],check=False)
        print("\nتم تشغيل فحص الوسائط القياسية: GLB للموديلات المدعومة، وبيانات codec للصوت، وأبعاد الخامات.")
        print("Manifest التكامل:",converted/"media-integration-manifest.json")
        mobile_apk_inspector = WORK / "tools/asset_forensics/inspect_mobile_apks.py"
        mobile_apk_output = destination / "mobile-apk-analysis"
        subprocess.run([sys.executable,str(mobile_apk_inspector),str(destination/"extracted/root"),"--out",str(mobile_apk_output)],check=True)
        mobile_apk_report = json.loads((mobile_apk_output/"mobile-apk-static-analysis.json").read_text(encoding="utf-8"))
        print("\\nتحليل APK/DEX:",mobile_apk_report["apk_count"],"حزمة APK /",mobile_apk_report["dex_count"],"ملف DEX /",mobile_apk_report["native_library_count"],"مكتبة أصلية")
        print("تقرير APK:",mobile_apk_output/"mobile-apk-static-analysis.json")
        native_inspector = WORK / "tools/asset_forensics/inspect_native_libraries.py"
        native_output = destination / "native-libraries"
        subprocess.run([sys.executable,str(native_inspector),str(destination/"extracted/root"),"--out",str(native_output)],check=True)
        native_report = json.loads((native_output/"native-library-inventory.json").read_text(encoding="utf-8"))
        print("\\nفحص مكتبات Android الأصلية:",native_report["status"])
        print("المكتبات الفريدة:",native_report["unique_libraries_extracted"])
        print("المعماريات:",native_report["architecture_counts"])
        print("تقرير المكتبات:",native_output/"native-library-inventory.json")
        print("المكتبات المستخرجة محليًا داخل مجلد النتائج الخاص:",native_output)
        fui_inspector = WORK / "tools/asset_forensics/inspect_fui_headers.py"
        fui_output = destination / "fui-analysis"
        subprocess.run([sys.executable,str(fui_inspector),str(destination/"extracted/root"),"--out",str(fui_output)],check=True)
        fui_header_report = json.loads((fui_output/"fui-header-inventory.json").read_text(encoding="utf-8"))
        print("\\nفحص ترويسات FUI:",fui_header_report["file_count"],"ملف")
        print("حزم FairyGUI V7 المرشحة:",fui_header_report["fairygui_v7_candidates"])
        if fui_header_report["fairygui_v7_candidates"]:
            import shutil
            fui_tool_dir = pathlib.Path("/content/openfairygui-parser")
            fui_tool_dir.mkdir(parents=True,exist_ok=True)
            node_path = shutil.which("node")
            node_version = subprocess.run([node_path,"--version"],capture_output=True,text=True) if node_path else None
            node_major = int(node_version.stdout.strip().lstrip("v").split(".")[0]) if node_version and node_version.returncode == 0 else 0
            npm = shutil.which("npm")
            if not npm:
                subprocess.run(["bash","-lc","apt-get install -y -qq nodejs npm"],check=True)
                npm = shutil.which("npm")
            packages = ["@openfairygui/core@0.7.0"] + (["node@22"] if node_major < 22 else [])
            install = subprocess.run([npm,"install","--prefix",str(fui_tool_dir),"--no-audit","--no-fund","--silent",*packages],text=True)
            if install.returncode == 0:
                parser_copy = fui_tool_dir/"parse_fui_packages.mjs"
                shutil.copy2(WORK/"tools/asset_forensics/parse_fui_packages.mjs",parser_copy)
                node_bin = fui_tool_dir/"node_modules/.bin/node" if node_major < 22 else shutil.which("node")
                if node_bin and pathlib.Path(node_bin).exists():
                    subprocess.run([str(node_bin),str(parser_copy),str(fui_output/"fui-header-inventory.json"),str(fui_output)],check=False)
                    parsed_report = fui_output/"fui-package-analysis.json"
                    if parsed_report.exists():
                        parsed_data=json.loads(parsed_report.read_text(encoding="utf-8"))
                        print("تحليل FairyGUI V7:",parsed_data["parsed"],"نجح /",parsed_data["failed"],"فشل")
                else:
                    print("لم يتم العثور على Node 22 لتشغيل محلل FairyGUI؛ تم الاحتفاظ بتقرير الترويسات.")
            else:
                print("تعذر تثبيت OpenFairyGUI؛ تم الاحتفاظ بتقرير الترويسات دون اعتبار الملفات غير متوافقة.")
        print("تقرير FUI:",fui_output/"fui-header-inventory.json")
        print("المرحلة التالية: تحليل C3 / skeleton / motion / FUI ببارسرات متحققة من بنية الملفات.")
        if report.exists():
            import json
            data=json.loads(report.read_text(encoding="utf-8"))
            print("\nملخص:")
            print("عدد الملفات:",data.get("file_count"))
            print("التصنيفات:",data.get("category_counts"))
            print("الأرشيفات:",data.get("archives"))
            print("أخطاء:",data.get("errors"))

        print('\\nبدء تجهيز نسختين من النتائج: Drive للعمل وGitHub Release للمراجعة والاستخدام.', flush=True)
        import hashlib, datetime, requests, getpass
        OUT_ROOT.mkdir(parents=True, exist_ok=True)
        source_sha = hashlib.sha256(source.read_bytes()).hexdigest()
        inventory = []
        for item in sorted(destination.rglob('*')):
            if item.is_file():
                inventory.append({'path': str(item.relative_to(destination)), 'size_bytes': item.stat().st_size})
        export_manifest = {'source_name': source.name, 'source_sha256': source_sha, 'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'drive_output': str(destination), 'file_count': len(inventory), 'total_bytes': sum(x['size_bytes'] for x in inventory), 'files': inventory}
        (destination/'export-manifest.json').write_text(json.dumps(export_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
        print('نسخة Drive جاهزة:', destination, '| ملفات:', len(inventory), flush=True)
        staging = pathlib.Path('/content/github-release-staging') / (source.stem + '-' + source_sha[:10])
        if staging.exists():
            import shutil
            shutil.rmtree(staging)
        staging.mkdir(parents=True, exist_ok=True)
        archive_jobs = [(p.name, [p.name]) for p in sorted(destination.iterdir()) if p.is_dir()]
        root_files = [p.name for p in destination.iterdir() if p.is_file()]
        if root_files:
            archive_jobs.append(('root-reports', root_files))
        if not archive_jobs:
            raise RuntimeError('لا توجد ملفات ناتجة لتجهيز نسخة GitHub.')
        for label, members in archive_jobs:
            archive_base = staging / (label.replace('/', '_') + '.7z')
            print('ضغط مجموعة GitHub:', label, flush=True)
            subprocess.run(['7z','a','-t7z','-mx=1','-v1800m',str(archive_base),*members], cwd=str(destination), check=True)
        token = None
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN')
        except Exception:
            token = None
        if not token:
            print('أدخلي GitHub Personal Access Token بصلاحية Contents: Read and write. الإدخال مخفي ولن يُحفظ في Drive.', flush=True)
            token = getpass.getpass('GitHub token: ')
        if not token:
            raise RuntimeError('لم يتم توفير GITHUB_TOKEN؛ نتائج Drive محفوظة، ويمكن إعادة خطوة النشر لاحقًا.')
        api = 'https://api.github.com/repos/ZzwahwahzZ/Clash-of-Legends'
        headers = {'Authorization': 'Bearer ' + token, 'Accept': 'application/vnd.github+json', 'X-GitHub-Api-Version': '2022-11-28'}
        tag = 'asset-forensics-' + source_sha[:10] + '-' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d%H%M%S')
        payload = {'tag_name': tag, 'name': 'Asset Forensics - ' + source.name, 'body': 'Automated forensic export. Complete extracted results are retained in Google Drive. GitHub assets are split into 1.8 GB 7z volumes.', 'draft': False, 'prerelease': True}
        response = requests.post(api + '/releases', headers=headers, json=payload, timeout=60)
        if response.status_code not in (200,201):
            raise RuntimeError('GitHub release creation failed: ' + str(response.status_code) + ' ' + response.text[:1000] + '. Drive copy is retained.')
        release = response.json()
        upload_base = release['upload_url'].split('{')[0]
        uploaded = []
        assets = sorted(p for p in staging.iterdir() if p.is_file())
        for asset in assets:
            print('رفع إلى GitHub Release:', asset.name, f'({asset.stat().st_size/1024/1024:.1f} MiB)', flush=True)
            with asset.open('rb') as stream:
                up = requests.post(upload_base, params={'name': asset.name}, headers={**headers, 'Content-Type': 'application/x-7z-compressed'}, data=stream, timeout=(60,3600))
            if up.status_code not in (200,201):
                raise RuntimeError('فشل رفع ' + asset.name + ': ' + str(up.status_code) + ' ' + up.text[:800] + '. النسخة الكاملة ما زالت محفوظة على Drive.')
            uploaded.append(up.json().get('browser_download_url'))
        print('اكتمل رفع نسخة GitHub:', release.get('html_url'), flush=True)
        print('عدد ملفات الأرشيف المرفوعة:', len(uploaded), flush=True)
        for url in uploaded:
            print(url, flush=True)
        print('Drive يحتفظ بمجلد النتائج الكامل، وGitHub Release يحتوي نسخ 7z مجزأة قابلة للتنزيل والفك. لن تُحذف نسخة Drive.', flush=True)
# واجهة اختيار تفاعلية: أبقي الخلية قيد التنفيذ حتى اختيار الملف والضغط على زر البدء.
import asyncio
import ipywidgets as widgets
from IPython.display import display, clear_output

chooser = FileChooser('/content/drive/MyDrive')
chooser.title = '<b>اختاري ملف كونكر الذي تريدين فحصه</b>'
chooser.filter_pattern = ['*.zip','*.7z','*.rar','*.tar','*.tgz','*.apk','*.obb']
chooser.show_only_dirs = False

run_button = widgets.Button(
    description='ابدئي الفحص والاستخراج',
    button_style='success',
    icon='search'
)
output = widgets.Output(layout={'border':'1px solid #888','padding':'8px'})
loop = asyncio.get_running_loop()
start_future = loop.create_future()

def on_run_clicked(_):
    selected = chooser.selected
    with output:
        clear_output(wait=True)
        if not selected:
            print('حددي ملفًا من المتصفح أولًا، ثم اضغطي زر البدء.', flush=True)
            return
        if not start_future.done():
            start_future.set_result(str(selected))

run_button.on_click(on_run_clicked)
display(chooser, run_button, output)
print('الخلية الآن في وضع انتظار نشط. اختاري الملف ثم اضغطي زر «ابدئي الفحص والاستخراج».', flush=True)

# await يترك حلقة أحداث Jupyter تعمل؛ لذلك تظل أزرار ipywidgets تستجيب
# بينما تبقى الخلية نفسها Running حتى يبدأ الفحص وينتهي.
selected_source = await start_future

with output:
    clear_output(wait=True)
    print('الملف المحدد:', selected_source, flush=True)
    print('بدأ التنفيذ داخل الخلية نفسها...', flush=True)
    try:
        run_pipeline(selected_source)
        print('انتهى تنفيذ خط الفحص.', flush=True)
    except Exception as exc:
        import traceback
        print('توقفت إحدى مراحل الفحص بسبب خطأ:', repr(exc), flush=True)
        traceback.print_exc()
